# Credit Scoring Prediction System 

**Objective:** Build an end-to-end credit scoring model and a professional Streamlit web app to predict whether an applicant is creditworthy (Good) or not (Bad).

**Problem Statement:** Given a person's financial and personal history, predict creditworthiness to support lending decisions.

**Real-world Use Case:** Banks and fintech companies can use this system to automate loan approvals, reduce defaults, and improve risk management.

---

### Notebook Structure
This notebook is organized into clear, executable cells for a demo video: imports, data creation/loading, EDA, preprocessing, feature engineering, model training, evaluation, model export, and Streamlit app generation.

In [ ]:
# Import Libraries
# Detailed imports placed in a single cell so the demo can show environment requirements.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Generate a Realistic Synthetic Dataset
We generate a synthetic dataset with realistic distributions so this notebook is runnable without external downloads. Columns include Age, Income, LoanAmount, CreditCardDebt, PaymentHistory, NumLoans, CreditUtilization, EmploymentStatus, ExistingDebts, Savings, and the target `CreditLabel` (Good/Bad).
We include seeded randomness for reproducibility.

In [ ]:
# Create synthetic dataset
np.random.seed(42)
n = 5000
age = np.random.normal(40, 12, n).clip(18, 80).astype(int)
income = np.random.lognormal(10.5, 0.7, n).astype(int)  # roughly thousands
loan_amount = np.random.normal(15000, 8000, n).clip(1000, 100000).astype(int)
credit_card_debt = np.random.normal(3000, 2500, n).clip(0, 50000).astype(int)
payment_history = np.random.choice(['Excellent','Good','Late','Default'], size=n, p=[0.5,0.3,0.15,0.05])
num_loans = np.random.poisson(1.5, n)
credit_util = np.random.beta(2,5, n)  # proportion
employment_status = np.random.choice(['Employed','Self-employed','Unemployed','Retired'], size=n, p=[0.7,0.15,0.1,0.05])
existing_debts = np.random.normal(10000, 7000, n).clip(0, 100000).astype(int)
savings = np.random.normal(8000, 10000, n).clip(0, 200000).astype(int)
# Heuristic credit score (not required but helpful)
score = (income/1000)*0.3 + (savings/1000)*0.2 - (loan_amount/1000)*0.25 - (existing_debts/1000)*0.15 + (num_loans * -1.0)
# adjust by payment history
hist_factor = np.array([1.2 if s=='Excellent' else 1.0 if s=='Good' else 0.7 if s=='Late' else 0.4 for s in payment_history])
score = score * hist_factor + np.random.normal(0, 5, n)
# Create label: threshold-based with noise
label = np.where(score > np.percentile(score, 55), 'Good', 'Bad')
data = pd.DataFrame({
    'Age': age,
    'Income': income,
    'LoanAmount': loan_amount,
    'CreditCardDebt': credit_card_debt,
    'PaymentHistory': payment_history,
    'NumLoans': num_loans,
    'CreditUtilization': (credit_util*100).round(2),
    'EmploymentStatus': employment_status,
    'ExistingDebts': existing_debts,
    'Savings': savings,
    'ScoreApprox': score.round(2),
    'CreditLabel': label
})
data.head()

## Exploratory Data Analysis (EDA)
We'll inspect the dataset, check missing values, distributions, correlations, and class balance. Each visualization is separated so you can execute step-by-step in a demo.

In [ ]:
# Dataset overview
print('Shape:', data.shape)
display(data.describe(include='all').T)

In [ ]:
# Missing values check
data.isnull().sum()

In [ ]:
# Correlation heatmap for numerical features
num_cols = ['Age','Income','LoanAmount','CreditCardDebt','NumLoans','CreditUtilization','ExistingDebts','Savings','ScoreApprox']
plt.figure(figsize=(10,8))
sns.heatmap(data[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Feature distributions (2x2 grid for demo)
plt.figure(figsize=(12,8))
plt.subplot(2,2,1)
sns.histplot(data['Income'], bins=40, kde=True, color='teal')
plt.title('Income Distribution')
plt.subplot(2,2,2)
sns.histplot(data['LoanAmount'], bins=40, kde=True, color='orange')
plt.title('Loan Amount Distribution')
plt.subplot(2,2,3)
sns.histplot(data['CreditCardDebt'], bins=40, kde=True, color='red')
plt.title('Credit Card Debt')
plt.subplot(2,2,4)
sns.histplot(data['Savings'], bins=40, kde=True, color='green')
plt.title('Savings Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for a subset to keep it fast
sns.pairplot(data[['Income','LoanAmount','CreditCardDebt','Savings','ScoreApprox','CreditLabel']], hue='CreditLabel', corner=True, plot_kws={'alpha':0.3})
plt.suptitle('Pairplot of key financial features', y=1.02)

In [ ]:
# Class balance chart
plt.figure(figsize=(6,4))
sns.countplot(x='CreditLabel', data=data, palette=['#2ecc71','#e74c3c'])
plt.title('Class Balance')
plt.show()

## Data Preprocessing
Handle missing values (none expected), encode categorical variables, scale numeric features, and split the data.

In [ ]:
# Encode categorical columns
df = data.copy()
# Label encode target
le = LabelEncoder()
df['CreditLabelEnc'] = le.fit_transform(df['CreditLabel'])
# PaymentHistory and EmploymentStatus -> one-hot or ordinal mapping
df = pd.get_dummies(df, columns=['PaymentHistory','EmploymentStatus'], drop_first=True)
# Drop helper ScoreApprox for modeling (we created it artificially)
df.drop(columns=['ScoreApprox','CreditLabel'], inplace=True)
df.head()

## Feature Engineering
Create derived features useful for credit assessment: Debt-to-Income ratio, Financial Stability Score, and Payment Consistency Score (derived from payment history flags).

In [ ]:
# Debt-to-income ratio
df['DebtToIncome'] = (df['ExistingDebts'] + df['LoanAmount'] + df['CreditCardDebt']) / (df['Income'] + 1)
# Financial stability: blend of savings, income, and number of loans
df['FinancialStability'] = (df['Savings']/1000)*0.4 + (df['Income']/1000)*0.4 - (df['NumLoans']*2)
# Payment consistency: map one-hot columns back to scores
# Create a simple consistency metric: Excellent=1.0, Good=0.8, Late=0.4, Default=0.1
df['PaymentConsistency'] = 0
if 'PaymentHistory_Good' in df.columns:
    df['PaymentConsistency'] = df['PaymentHistory_Excellent']*1.0 + df['PaymentHistory_Good']*0.8 + df.get('PaymentHistory_Late',0)*0.4 + df.get('PaymentHistory_Default',0)*0.1
# Drop individual PaymentHistory columns optionally
for col in list(df.columns):
    if col.startswith('PaymentHistory_'):
        try:
            df.drop(columns=[col], inplace=True)
        except Exception:
            pass
df.head()

In [ ]:
# Prepare features and target
X = df.drop(columns=['CreditLabelEnc'])
y = df['CreditLabelEnc']
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# Feature scaling for numeric columns
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])
print('Training samples:', X_train.shape[0])

## Model Building
Train Logistic Regression, Decision Tree, and Random Forest. Keep training fast by limiting hyperparameter search.

In [ ]:
# Train models
models = {}
# Logistic Regression
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
models['LogisticRegression'] = lr
# Decision Tree
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
models['DecisionTree'] = dt
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['RandomForest'] = rf
print('Trained models:', list(models.keys()))

## Model Evaluation
Evaluate each model with standard metrics and display confusion matrices and ROC curves.

In [ ]:
# Evaluate and compare models
results = []
plt.figure(figsize=(8,6))
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    results.append({'Model':name,'Accuracy':acc,'Precision':prec,'Recall':rec,'F1':f1,'ROC_AUC':roc})
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()
results_df = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False)
results_df

## Best Model Selection and Saving
Choose the best model by ROC_AUC and save it with the scaler and encoders so the Streamlit app can load it directly.

In [ ]:
# Select best model by ROC_AUC from results
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
print('Best model:', best_name)
# Save model, scaler, and label encoder together
artifact = {'model': best_model, 'scaler': scaler, 'label_encoder': le, 'feature_columns': X.columns.tolist()}
os.makedirs('artifacts', exist_ok=True)
joblib.dump(artifact, 'artifacts/model.pkl')
print('Saved artifact to artifacts/model.pkl')

## (Optional) SHAP Feature Importance
If SHAP is installed, compute and plot feature importance for the best model. This is a bonus internship-ready explanation feature.

In [ ]:
# SHAP explanation (optional)
try:
    import shap
    explainer = shap.TreeExplainer(best_model) if hasattr(best_model, 'estimators_') or isinstance(best_model, RandomForestClassifier) else shap.Explainer(best_model, X_train)
    shap_values = explainer.shap_values(X_train[:200])
    plt.title('SHAP Feature Importance (summary)')
    shap.summary_plot(shap_values, X_train.iloc[:200], show=True)
except Exception as e:
    print('SHAP not available or failed:', e)

## Generate Streamlit App Files Automatically
We'll write `app.py`, `requirements.txt`, and `README.md` to the workspace so you can run the web app immediately after running this notebook (it also already saved `artifacts/model.pkl`).

In [ ]:
# Write Streamlit app to disk
app_code = r
'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from PIL import Image
import time
# Load artifact
artifact = joblib.load('artifacts/model.pkl')
model = artifact['model']
scaler = artifact['scaler']
feature_columns = artifact['feature_columns']
st.set_page_config(page_title='Credit Scoring Dashboard', layout='wide', initial_sidebar_state='expanded')
# Custom CSS for gradient background and card styling
st.markdown('<style>
.reportview-container {background: linear-gradient(120deg, #0f2027, #203a43, #2c5364);} 
section.main {background: rgba(255,255,255,0.03); padding: 2rem; border-radius:12px;}
.stButton>button{background:linear-gradient(90deg,#0072ff,#00c6ff); color:white;}
</style>
,
,
1
2
,
18
80
35
,
,
,
,

10
1
,

100
20
,
,
,
,
,
,
,